In [1]:
import pandas as pd
import requests
import time

from pandas import json_normalize

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 24.86, "longitude": 67.01,"hourly": "temperature_2m,relative_humidity_2m",
    "timezone": "Asia/Karachi",
}
r = requests.get(url, params= params, timeout=10)
r.raise_for_status()
data = r.json()

In [2]:
r.status_code
r.headers["content-type"]
list(data.keys())
data["hourly"].keys()

dict_keys(['time', 'temperature_2m', 'relative_humidity_2m'])

In [3]:
df = pd.DataFrame(data["hourly"])
df["time"] = pd.to_datetime(df["time"])
df.dtypes

time                    datetime64[us]
temperature_2m                 float64
relative_humidity_2m             int64
dtype: object

In [4]:
cities = {"Karachi": (24.86, 67.01), "Lahore": (31.55, 74.34), "Riyadh": (24.71, 46.68), "Dubai": (25.20, 55.27),"Dammam": (26.43, 50.10)}
frames = []

for name, (lat,lon) in cities.items():
    params = {
        "latitude" : lat,
        "longitude" : lon,
        "hourly": "temperature_2m,relative_humidity_2m",
        "timezone": "Asia/Karachi"
    }
    try:
        r = requests.get(url, params= params, timeout= 10)
        r.raise_for_status()
        data = r.json()
    except requests.RequestException as e:
        print(e)
        continue

    df = pd.DataFrame(data["hourly"])
    df["time"] = pd.to_datetime(df["time"])
    df["city"] = name

    frames.append(df)
    time.sleep(1)


combined = pd.concat(frames, ignore_index=True)
print(combined.shape)
print(combined["city"].value_counts())


(840, 4)
city
Karachi    168
Lahore     168
Riyadh     168
Dubai      168
Dammam     168
Name: count, dtype: int64


In [5]:

combined.groupby("city")["temperature_2m"].agg(["min", "max","mean"]).round(1)

,min,max,mean
city,,,
Dammam,27.4,45.8,36.2
Dubai,28.4,43.7,34.7
Karachi,25.1,36.7,28.5
Lahore,26.3,36.9,31.6
Riyadh,29.2,44.1,37.8


In [6]:
nested = [{"id": 1, "user": {"name": "Sara", "addr": {"city": "Karachi"}}}]

flat = pd.json_normalize(nested)
print(flat)
print("columns:", list(flat.columns))


   id user.name user.addr.city
0   1      Sara        Karachi
columns: ['id', 'user.name', 'user.addr.city']
